# 06 — Vision-language model (VLM) baseline: Qwen2-VL-2B
Answers the reviewer question *"why not just use a VLM?"*

| Part | What | Time (RTX 3050) |
|---|---|---|
| A | **Zero-shot** Qwen2-VL-2B-Instruct (bf16): image + meme text in the prompt, answer scored over the label options. Raw and **calibrated** (contextual calibration on validation memes) | ~30–45 min |
| B | **QLoRA fine-tuning** (4-bit, LoRA r=16 on the language model), 1 epoch, trained on a 50/50 mix of gold and OCR text | ~3 h for 1 seed |
| C | Comparison with NAFIM and the mixed-text fusion baseline (notebook 04) | seconds |

Tasks: BHM (hate), MIMOSA (5-class + binary), BanglaAbuseMeme (abuse, OCR text only).
Prediction = the label option with the highest next-token probability (A–E letters for 5-class, No/Yes for binary) — no free-text parsing.

**Before running** (internship1 env) — pinned versions that keep torch 2.5.1 / transformers 4.46.3:
```bash
pip install "peft==0.13.2" "bitsandbytes==0.44.1" "accelerate==1.0.1"
python -c "import torch, transformers; print(torch.__version__, transformers.__version__)"   # must still be 2.5.1 / 4.46.3
```
First run downloads Qwen2-VL-2B-Instruct (~4.5 GB). Run **after notebook 05** finishes (same GPU). Resumable.

## 1. Imports & config

In [ ]:
import os, gc, json, time, random, platform
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
import transformers, sklearn
from sklearn.metrics import f1_score, accuracy_score

ROOT = "/mnt/f415dcc8-27a5-4e66-8493-25f1d939974c/internshipwork/journal"
P = f"{ROOT}/processed"
NB04 = f"{ROOT}/results/04_nafim"
RES = f"{ROOT}/results/06_vlm"
PRED, ADAPT = f"{RES}/preds", f"{RES}/adapters"
os.makedirs(PRED, exist_ok=True); os.makedirs(ADAPT, exist_ok=True)

VLM_NAME = "Qwen/Qwen2-VL-2B-Instruct"
MAX_PIXELS = 448 * 448          # caps image tokens (~256 per image) to fit 8 GB
MIN_PIXELS = 128 * 28 * 28
MAX_TEXT_CHARS = 300            # meme text is truncated in the prompt
CAL_N = 300                     # validation memes used for contextual calibration (zero-shot)

RUN_ZERO_SHOT, RUN_QLORA = True, True
QLORA_SEEDS = [42]
QLORA = dict(r=16, alpha=32, dropout=0.05, lr=2e-4, epochs=1, grad_accum=8, warmup=0.03,
             target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
MAX_TRAIN = None                # e.g. 2000 to subsample training memes and save time

TEXT_COLS = {"bhm": {"gold": "text", "ocr": "text_ocr"},
             "mimosa": {"gold": "text", "ocr": "text_ocr"},
             "banglaabusememe": {"ocr": "text"}}
TASKS = {
    ("bhm", "label"): dict(q="Is this meme hateful, i.e. does it attack or demean a person or group?",
                           opts=["No", "Yes"]),
    ("mimosa", "label"): dict(q="Which type of aggression does this meme contain?\n"
                                "A) none  B) political  C) religious  D) gendered  E) other",
                              opts=["A", "B", "C", "D", "E"]),
    ("mimosa", "label_bin"): dict(q="Is this meme aggressive?", opts=["No", "Yes"]),
    ("banglaabusememe", "label"): dict(q="Is this meme abusive?", opts=["No", "Yes"]),
}
dev = "cuda"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| transformers", transformers.__version__)
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 2**30:.2f} / {total / 2**30:.2f} GiB")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## 2. Data and prompts

In [ ]:
data = {ds: pd.read_csv(f"{P}/{ds}.csv").fillna("") for ds in TEXT_COLS}
for ds, df in data.items():
    for c in TEXT_COLS[ds].values():
        df[c] = df[c].astype(str)
    print(ds, df.shape, "| test:", (df["split"] == "test").sum())

def idx_of(df, s):
    return np.where(df["split"].values == s)[0]

def build_prompt(task, text):
    spec = TASKS[task]
    t = " ".join(str(text).split())[:MAX_TEXT_CHARS] or "(no text detected)"
    return ("This is a Bengali social-media meme. The text written on the meme "
            "(automatically extracted, may contain errors) is:\n"
            f"«{t}»\n\n{spec['q']}\nAnswer with only one of: {', '.join(spec['opts'])}.")

print(build_prompt(("mimosa", "label"), data["mimosa"]["text"].iloc[0]))

## 3. Model loading and option scoring

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

processor = AutoProcessor.from_pretrained(VLM_NAME, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
tokenizer = processor.tokenizer

OPT_IDS = {}
for task, spec in TASKS.items():
    ids = [tokenizer.encode(o, add_special_tokens=False)[0] for o in spec["opts"]]
    assert len(set(ids)) == len(ids), f"option first-tokens collide for {task}: {spec['opts']}"
    OPT_IDS[task] = torch.tensor(ids)
print({k: v.tolist() for k, v in OPT_IDS.items()})

def load_vlm(four_bit=False):
    kw = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_use_double_quant=True,
                                                       bnb_4bit_compute_dtype=torch.bfloat16)
    return Qwen2VLForConditionalGeneration.from_pretrained(VLM_NAME, **kw)

def make_inputs(task, image_path, text):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": build_prompt(task, text)}]}]
    chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img = Image.open(image_path).convert("RGB")
    return processor(text=[chat], images=[img], return_tensors="pt").to(dev)

def option_logits(model, task, image_path, text):
    """Logits of the label options at the answer position, shape [n_options] (float32)."""
    out = model(**make_inputs(task, image_path, text))
    return out.logits[0, -1, OPT_IDS[task].to(dev)].float()

@torch.no_grad()
def score_rows(model, task, rows, text_col, desc):
    model.eval(); df = data[task[0]]; probs = []
    for i in tqdm(rows, desc=desc, leave=False):
        probs.append(torch.softmax(option_logits(model, task, df["image_path"].iloc[i], df[text_col].iloc[i]), -1).cpu().numpy())
    return np.stack(probs)

## 4. Part A — zero-shot (raw + contextual calibration)

In [ ]:
def calibrate(p_test, p_cal):
    """Contextual calibration: divide by the model's mean option probability on validation memes."""
    q = p_test / np.maximum(p_cal.mean(0, keepdims=True), 1e-8)
    return q / q.sum(1, keepdims=True)

if RUN_ZERO_SHOT:
    model = None
    for task in TASKS:
        ds, lab = task; df = data[ds]; te = idx_of(df, "test")
        rng = np.random.default_rng(0); va = idx_of(df, "val"); cal = rng.choice(va, min(CAL_N, len(va)), replace=False)
        for src, col in TEXT_COLS[ds].items():
            f_te, f_cal = f"{PRED}/zs__{ds}__{lab}__test-{src}.npy", f"{PRED}/zs__{ds}__{lab}__cal-{src}.npy"
            if os.path.exists(f_te) and os.path.exists(f_cal):
                continue
            if model is None:
                model = load_vlm(four_bit=False)
            t0 = time.time()
            np.save(f_cal, score_rows(model, task, cal, col, f"zs cal {ds}/{lab}/{src}"))
            np.save(f_te, score_rows(model, task, te, col, f"zs test {ds}/{lab}/{src}"))
            print(f"zero-shot {ds}/{lab}/{src} done in {(time.time() - t0) / 60:.1f} min")
    if model is not None:
        del model; gc.collect(); torch.cuda.empty_cache()

zs_rows = []
for task in TASKS:
    ds, lab = task; df = data[ds]; y = df[lab].values[idx_of(df, "test")].astype(int)
    for src in TEXT_COLS[ds]:
        f_te, f_cal = f"{PRED}/zs__{ds}__{lab}__test-{src}.npy", f"{PRED}/zs__{ds}__{lab}__cal-{src}.npy"
        if not os.path.exists(f_te):
            continue
        p, pc = np.load(f_te), np.load(f_cal)
        for kind, pp in [("zero-shot (raw)", p), ("zero-shot (calibrated)", calibrate(p, pc))]:
            pred = pp.argmax(1)
            zs_rows.append({"dataset": ds, "task": lab, "test_text": src, "model": f"Qwen2-VL-2B {kind}",
                            "macro_f1": f1_score(y, pred, average="macro"), "acc": accuracy_score(y, pred),
                            "pred_class_dist": np.bincount(pred, minlength=p.shape[1]).tolist()})
zs_df = pd.DataFrame(zs_rows)
zs_df

## 5. Part B — QLoRA fine-tuning (verbalizer classification over the option tokens)

In [ ]:
def run_qlora(task, seed):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import get_linear_schedule_with_warmup
    set_seed(seed)
    ds, lab = task; df = data[ds]; y = df[lab].values.astype(int); n = len(TASKS[task]["opts"])
    tr, va, te = [idx_of(df, s) for s in ("train", "val", "test")]
    if MAX_TRAIN:
        tr = np.random.default_rng(seed).choice(tr, min(MAX_TRAIN, len(tr)), replace=False)
    counts = np.bincount(y[tr], minlength=n)
    cw = torch.tensor(len(tr) / (n * np.maximum(counts, 1)), dtype=torch.float32, device=dev)

    model = load_vlm(four_bit=True)
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(r=QLORA["r"], lora_alpha=QLORA["alpha"], lora_dropout=QLORA["dropout"],
                                             target_modules=QLORA["target_modules"], bias="none", task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=QLORA["lr"], weight_decay=0.0)
    total_steps = int(np.ceil(len(tr) * QLORA["epochs"] / QLORA["grad_accum"]))
    sch = get_linear_schedule_with_warmup(opt, int(QLORA["warmup"] * total_steps), total_steps)
    cols = TEXT_COLS[ds]; rng = np.random.default_rng(seed)

    model.train(); step = 0
    for ep in range(QLORA["epochs"]):
        order = rng.permutation(tr); opt.zero_grad(set_to_none=True); run_loss = 0.0
        pbar = tqdm(order, desc=f"qlora {ds}/{lab} s{seed} ep{ep + 1}", leave=False)
        for k, i in enumerate(pbar):
            col = cols["ocr"] if len(cols) == 1 or rng.random() < 0.5 else cols["gold"]   # 50/50 gold / OCR text
            lo = option_logits(model, task, df["image_path"].iloc[i], df[col].iloc[i])
            loss = F.cross_entropy(lo[None], torch.tensor([y[i]], device=dev), weight=cw) / QLORA["grad_accum"]
            loss.backward(); run_loss += loss.item()
            if (k + 1) % QLORA["grad_accum"] == 0 or k + 1 == len(order):
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True); step += 1
                if step % 20 == 0:
                    pbar.set_postfix(loss=f"{run_loss * QLORA['grad_accum'] / ((k + 1)):.3f}")
    out = {"val": {}, "test": {}}
    for src, col in cols.items():
        out["val"][src] = score_rows(model, task, va, col, f"qlora val {ds}/{lab}/{src}")
        out["test"][src] = score_rows(model, task, te, col, f"qlora test {ds}/{lab}/{src}")
    model.save_pretrained(f"{ADAPT}/{ds}__{lab}__s{seed}")
    del model, opt, sch; gc.collect(); torch.cuda.empty_cache()
    return out

if RUN_QLORA:
    for seed in QLORA_SEEDS:
        for task in TASKS:
            ds, lab = task
            files = {src: f"{PRED}/qlora__{ds}__{lab}__s{seed}__test-{src}.npy" for src in TEXT_COLS[ds]}
            if all(os.path.exists(f) for f in files.values()):
                continue
            t0 = time.time()
            out = run_qlora(task, seed)
            for src in TEXT_COLS[ds]:
                np.save(files[src], out["test"][src])
                np.save(f"{PRED}/qlora__{ds}__{lab}__s{seed}__val-{src}.npy", out["val"][src])
            print(f"QLoRA {ds}/{lab} seed {seed} done in {(time.time() - t0) / 60:.1f} min")

ql_rows = []
for task in TASKS:
    ds, lab = task; df = data[ds]; y = df[lab].values[idx_of(df, "test")].astype(int)
    for seed in QLORA_SEEDS:
        for src in TEXT_COLS[ds]:
            f = f"{PRED}/qlora__{ds}__{lab}__s{seed}__test-{src}.npy"
            if os.path.exists(f):
                pred = np.load(f).argmax(1)
                ql_rows.append({"dataset": ds, "task": lab, "test_text": src, "model": "Qwen2-VL-2B QLoRA", "seed": seed,
                                "macro_f1": f1_score(y, pred, average="macro"), "acc": accuracy_score(y, pred)})
ql_df = pd.DataFrame(ql_rows)
ql_df

## 6. Part C — comparison with NAFIM and the mixed-text fusion baseline (macro-F1)

In [ ]:
r04 = pd.read_csv(f"{NB04}/runs.csv").drop_duplicates(["train_id", "test_text"])
ours = (r04[r04.test_text.isin(["gold", "ocr"])]
        .groupby(["dataset", "task", "test_text", "model"])["test_f1"].agg(["mean", "std", "count"]).reset_index())
ours["macro_f1"] = ours.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}" if r["count"] > 1 else f"{r['mean']:.4f}", axis=1)

vlm = pd.concat([zs_df[["dataset", "task", "test_text", "model", "macro_f1"]] if len(zs_df) else None,
                 ql_df.groupby(["dataset", "task", "test_text", "model"])["macro_f1"].mean().reset_index() if len(ql_df) else None])
vlm["macro_f1"] = vlm["macro_f1"].map(lambda v: f"{v:.4f}")
comp = pd.concat([vlm, ours[["dataset", "task", "test_text", "model", "macro_f1"]]])
order = ["Qwen2-VL-2B zero-shot (raw)", "Qwen2-VL-2B zero-shot (calibrated)", "Qwen2-VL-2B QLoRA", "Fusion (mixed text)", "NAFIM"]
comp_tab = comp.pivot_table(index=["dataset", "task", "test_text"], columns="model", values="macro_f1", aggfunc="first")
comp_tab = comp_tab[[c for c in order if c in comp_tab.columns]]
print("NAFIM / Fusion: mean ± std over 5 seeds (notebook 04); QLoRA:", len(QLORA_SEEDS), "seed(s); zero-shot: deterministic")
comp_tab

## 7. Save results + config

In [ ]:
zs_df.to_csv(f"{RES}/A_zero_shot.csv", index=False)
ql_df.to_csv(f"{RES}/B_qlora.csv", index=False)
comp_tab.to_csv(f"{RES}/C_vlm_vs_nafim.csv")
config = {
    "model": VLM_NAME, "max_pixels": MAX_PIXELS, "min_pixels": MIN_PIXELS, "max_text_chars": MAX_TEXT_CHARS,
    "prompt_template": build_prompt(("bhm", "label"), "<meme text>"), "tasks": {f"{k[0]}/{k[1]}": v for k, v in TASKS.items()},
    "scoring": "softmax over next-token logits of the option tokens at the answer position",
    "zero_shot": {"precision": "bf16", "calibration": f"contextual calibration, {CAL_N} random validation memes (seed 0)"},
    "qlora": {**QLORA, "seeds": QLORA_SEEDS, "quantization": "4-bit nf4, double quant, bf16 compute",
              "training_text": "50/50 gold / OCR (BanglaAbuseMeme: OCR only)", "loss": "class-weighted CE over option logits",
              "max_train": MAX_TRAIN},
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
                 "sklearn": sklearn.__version__, "gpu": torch.cuda.get_device_name(0)},
}
try:
    import peft, bitsandbytes
    config["versions"].update({"peft": peft.__version__, "bitsandbytes": bitsandbytes.__version__})
except ImportError:
    pass
with open(f"{RES}/config.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print("saved to", RES)